In [ ]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (18, 8),
        'axes.labelsize': 'medium',
        'axes.titlesize': 'large',
        'xtick.labelsize': 'medium',
        'ytick.labelsize': 'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import datetime
import pytz
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

NYC = pytz.timezone('America/New_York')

import sys
sys.path.append('../../')

# SFR Fly RV Backtest

Customizable backtest for SOFR 3M-STIR calendar spread fly trades using the `SFRCalSpreadRV` screener.

**Strategy:** Enter flies when z-score exceeds threshold, exit on mean-reversion, stop-loss, take-profit, or max holding period. Barnes-style carry filter optionally required.

**Tweak the `BACKTEST_CONFIG` dict below to explore different parameterizations.**

---
## 1. Configuration

All tunable parameters in one place. Change these and re-run.

In [ ]:
BACKTEST_CONFIG = {
    # ── Data ──────────────────────────────────────────────────
    'data_start': '2024-06-01',        # lookback start (need history for z-score warmup)
    'bt_start':   '2025-01-02',        # backtest P&L starts here
    'bt_end':     'live',              # 'live' or '2026-03-31'
    'n_contracts': 12,                 # Q12 SOFR ladder depth
    'constant_maturity': True,         # use CM rank labels (SFR1, SFR2, ...)
    'roll_adjusted': True,             # smooth roll discontinuities

    # ── Structure ─────────────────────────────────────────────
    'fly_gap': 1,                      # 1=3M fly, 2=6M fly, 3=9M fly, 4=12M fly

    # ── Z-Score / Vol Windows ─────────────────────────────────
    'zscore_window': 60,               # rolling window for z-score (business days)
    'vol_window': 20,                  # rolling window for realized vol

    # ── Entry Triggers (ALL must be met) ──────────────────────
    'entry_min_zscore': 1.5,           # min |z-score| to enter
    'entry_require_carry': False,      # require carry aligned with direction?
    'entry_min_risk_adj_roll': 0.0,    # min |risk-adj roll| if carry filter on
    'entry_max_vol': None,             # max annualized vol to enter (None=no filter)

    # ── Exit Triggers (first match wins) ──────────────────────
    # Mean reversion: exit when z-score crosses zero
    'exit_mean_reversion': True,

    # Take profit: exit when z-score falls below this (absolute)
    'exit_take_profit_zscore': None,   # e.g., 0.5 = exit when |z| < 0.5

    # Take profit: exit after earning N bp
    'exit_take_profit_bp': None,       # e.g., 3.0 = exit after +3bp realized

    # Stop-loss: exit when z-score worsens by this many sigma
    'exit_stop_loss_sd': 2.0,          # exit if |z| grows by 2 from entry

    # Stop-loss: exit after losing N bp
    'exit_stop_loss_bp': None,         # e.g., -5.0 = stop at -5bp loss

    # Max holding: force exit after N business days
    'exit_max_holding_days': 22,       # ~1 month

    # Time decay exit: exit if carry erodes edge
    'exit_carry_decay': False,         # exit when remaining carry < cost

    # ── Portfolio Rules ────────────────────────────────────────
    'max_concurrent_trades': 3,        # max open flies at once (None=unlimited)
    'no_duplicate_flies': True,        # don't re-enter same fly while open
    'reentry_after_stop': True,        # allow re-entry after stop-loss?
    'reentry_cooldown_days': 5,        # wait N days after stop before re-entry

    # ── Sizing & Costs ────────────────────────────────────────
    'belly_bpv': 100_000,              # notional risk per trade ($)
    'round_trip_cost_bp': 0.5,         # transaction cost in bps
}

print('Config loaded.')
for section in ['Data', 'Structure', 'Entry', 'Exit', 'Portfolio', 'Sizing']:
    print(f'\n  [{section}]')
    prefix = section.lower()[:4]
    for k, v in BACKTEST_CONFIG.items():
        if (section == 'Data' and k.startswith(('data', 'bt_', 'n_con', 'const', 'roll'))) or \
           (section == 'Structure' and k.startswith('fly')) or \
           (section == 'Entry' and k.startswith('entry')) or \
           (section == 'Exit' and k.startswith('exit')) or \
           (section == 'Portfolio' and k.startswith(('max_', 'no_', 'reent'))) or \
           (section == 'Sizing' and k.startswith(('belly', 'round'))):
            print(f'    {k:30s} = {v}')

---
## 2. Load Data & Compute Signals

In [ ]:
from BT.signals.sfr_cal_spread_rv import (
    SFRCalSpreadRVConfig,
    StructureType,
    STRUCTURE_LABELS,
    load_rate_panel,
    compute_fly_curve,
    compute_zscore_ts,
    compute_roll,
    compute_realized_vol,
    compute_risk_adj_roll,
    cm_label,
    resolve_cm_to_specific,
)
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.TimeseriesBuilder import TimeseriesBuilder

C = BACKTEST_CONFIG

sfr_config = SFRCalSpreadRVConfig(
    n_contracts=C['n_contracts'],
    zscore_window=C['zscore_window'],
    vol_window=C['vol_window'],
    constant_maturity=C['constant_maturity'],
    roll_adjusted=C['roll_adjusted'],
)

curve_mdp = IRSwapsMDP(source=sfr_config.source)
ts_builder = TimeseriesBuilder()

start = NYC.localize(datetime.datetime.fromisoformat(C['data_start']).replace(hour=18))
end = C['bt_end']

print('Loading rate panel...')
rates = load_rate_panel(sfr_config, start=start, end=end,
                        curve_mdp=curve_mdp, ts_builder=ts_builder)
print(f'  {rates.shape[0]} dates x {rates.shape[1]} contracts')
print(f'  Columns: {list(rates.columns)}')
print(f'  Range: {rates.index[0]} -> {rates.index[-1]}')
rates.tail(3)

In [ ]:
gap = C['fly_gap']
gap_label = {1: '3M', 2: '6M', 3: '9M', 4: '12M'}[gap]

# Compute fly curve (full time series)
fly_ts = compute_fly_curve(rates, gap=gap)
print(f'{gap_label} Fly curve: {fly_ts.shape[0]} dates x {fly_ts.shape[1]} flies')
print(f'Flies: {list(fly_ts.columns)}')

# Z-score time series
zscore_ts = compute_zscore_ts(fly_ts, window=C['zscore_window'])

# Vol time series (rolling)
vol_ts = fly_ts.diff().rolling(C['vol_window'], min_periods=10).std() * np.sqrt(252)

# Roll/carry at each date (computed from the cross-section)
# For backtest we need roll at every date, not just the latest
roll_ts = pd.DataFrame(np.nan, index=fly_ts.index, columns=fly_ts.columns)
for i in range(len(fly_ts)):
    row = fly_ts.iloc[i]
    for j in range(1, len(fly_ts.columns)):
        roll_ts.iloc[i, j] = row.iloc[j - 1] - row.iloc[j]

risk_adj_roll_ts = roll_ts / vol_ts.replace(0, np.nan)

print(f'\nSignals computed. Z-score NaN count (should decrease over warmup):')
print(f'  First date with valid z-scores: {zscore_ts.dropna(how="all").index[0]}')
print(f'  Backtest starts at: {C["bt_start"]}')

---
## 3. Backtest Engine

In [ ]:
@dataclass
class FlyTrade:
    """A single fly trade record."""
    fly_id: str
    entry_date: pd.Timestamp
    entry_level: float
    entry_zscore: float
    entry_vol: float
    entry_roll: float
    direction: int  # +1 = buy belly (expect fly to rise), -1 = sell belly
    # Exit
    exit_date: Optional[pd.Timestamp] = None
    exit_reason: Optional[str] = None
    exit_level: Optional[float] = None
    exit_zscore: Optional[float] = None
    realized_pnl: float = 0.0
    peak_pnl: float = 0.0
    trough_pnl: float = 0.0
    daily_pnls: list = field(default_factory=list)


def run_fly_backtest(
    fly_ts: pd.DataFrame,
    zscore_ts: pd.DataFrame,
    vol_ts: pd.DataFrame,
    roll_ts: pd.DataFrame,
    risk_adj_roll_ts: pd.DataFrame,
    config: dict,
) -> Tuple[List[FlyTrade], pd.Series]:
    """Run the fly backtest.

    Returns (trades, daily_pnl_series).
    """
    bt_start = pd.Timestamp(config['bt_start'], tz=NYC)
    fly_ids = list(fly_ts.columns)
    dates = fly_ts.index

    # Find backtest start index
    bt_mask = dates >= bt_start
    if not bt_mask.any():
        raise ValueError(f'No dates >= {bt_start}')
    bt_dates = dates[bt_mask]

    open_trades: List[FlyTrade] = []
    closed_trades: List[FlyTrade] = []
    daily_pnl = pd.Series(0.0, index=bt_dates)
    stopped_flies: Dict[str, pd.Timestamp] = {}  # fly_id -> stop date

    cost = config['round_trip_cost_bp']

    for dt in bt_dates:
        if dt not in fly_ts.index:
            continue
        prev_idx = fly_ts.index.get_loc(dt) - 1
        if prev_idx < 0:
            continue

        # ── Phase 1: MTM + Exit checks ──
        still_open = []
        for trade in open_trades:
            col = trade.fly_id
            if col not in fly_ts.columns:
                continue

            current_level = fly_ts.loc[dt, col]
            prev_level = fly_ts.iloc[prev_idx][col]
            current_z = zscore_ts.loc[dt, col] if dt in zscore_ts.index else np.nan

            if np.isnan(current_level) or np.isnan(prev_level):
                still_open.append(trade)
                continue

            # Daily P&L: direction * change in fly level (already in bp)
            day_pnl = trade.direction * (current_level - prev_level)
            trade.daily_pnls.append(day_pnl)
            trade.realized_pnl = sum(trade.daily_pnls)
            trade.peak_pnl = max(trade.peak_pnl, trade.realized_pnl)
            trade.trough_pnl = min(trade.trough_pnl, trade.realized_pnl)
            daily_pnl[dt] += day_pnl

            # Check exits
            holding_days = len(pd.bdate_range(trade.entry_date, dt)) - 1
            exit_reason = None

            # 1. Mean reversion: z-score crossed zero
            if config['exit_mean_reversion'] and not np.isnan(current_z):
                if trade.direction == 1 and current_z >= 0:
                    exit_reason = 'mean_reversion'
                elif trade.direction == -1 and current_z <= 0:
                    exit_reason = 'mean_reversion'

            # 2. Take profit: z-score fell below threshold
            if not exit_reason and config.get('exit_take_profit_zscore') and not np.isnan(current_z):
                if abs(current_z) < config['exit_take_profit_zscore']:
                    exit_reason = 'take_profit_zscore'

            # 3. Take profit: earned enough bp
            if not exit_reason and config.get('exit_take_profit_bp'):
                if trade.realized_pnl >= config['exit_take_profit_bp']:
                    exit_reason = 'take_profit_bp'

            # 4. Stop-loss: z-score worsened
            if not exit_reason and config.get('exit_stop_loss_sd') and not np.isnan(current_z):
                z_worsening = abs(current_z) - abs(trade.entry_zscore)
                if z_worsening > config['exit_stop_loss_sd']:
                    exit_reason = 'stop_loss_zscore'

            # 5. Stop-loss: lost too many bp
            if not exit_reason and config.get('exit_stop_loss_bp'):
                if trade.realized_pnl <= config['exit_stop_loss_bp']:
                    exit_reason = 'stop_loss_bp'

            # 6. Max holding
            if not exit_reason and config.get('exit_max_holding_days'):
                if holding_days >= config['exit_max_holding_days']:
                    exit_reason = 'max_holding'

            if exit_reason:
                trade.exit_date = dt
                trade.exit_reason = exit_reason
                trade.exit_level = current_level
                trade.exit_zscore = current_z
                trade.realized_pnl -= cost  # deduct round-trip cost
                closed_trades.append(trade)
                if 'stop' in exit_reason:
                    stopped_flies[trade.fly_id] = dt
            else:
                still_open.append(trade)

        open_trades = still_open

        # ── Phase 2: Entry checks ──
        open_fly_ids = {t.fly_id for t in open_trades}
        n_open = len(open_trades)

        for col in fly_ids:
            # Portfolio filters
            if config.get('max_concurrent_trades') and n_open >= config['max_concurrent_trades']:
                break
            if config.get('no_duplicate_flies') and col in open_fly_ids:
                continue
            # Cooldown after stop
            if col in stopped_flies:
                days_since_stop = len(pd.bdate_range(stopped_flies[col], dt)) - 1
                if not config.get('reentry_after_stop', True):
                    continue
                if days_since_stop < config.get('reentry_cooldown_days', 0):
                    continue

            z = zscore_ts.loc[dt, col] if dt in zscore_ts.index else np.nan
            level = fly_ts.loc[dt, col]
            vol = vol_ts.loc[dt, col] if dt in vol_ts.index else np.nan
            roll = roll_ts.loc[dt, col] if dt in roll_ts.index else np.nan
            radj = risk_adj_roll_ts.loc[dt, col] if dt in risk_adj_roll_ts.index else np.nan

            if np.isnan(z) or np.isnan(level):
                continue

            # Entry z-score threshold
            if abs(z) < config['entry_min_zscore']:
                continue

            # Vol filter
            if config.get('entry_max_vol') and not np.isnan(vol):
                if vol > config['entry_max_vol']:
                    continue

            # Direction: z < 0 = fly is cheap -> buy belly (+1)
            #            z > 0 = fly is rich  -> sell belly (-1)
            direction = -1 if z > 0 else 1

            # Carry filter: require carry aligned with direction
            if config.get('entry_require_carry') and not np.isnan(roll):
                carry_aligned = (direction == 1 and roll > 0) or (direction == -1 and roll < 0)
                if not carry_aligned:
                    continue
                if config.get('entry_min_risk_adj_roll', 0) > 0 and not np.isnan(radj):
                    if abs(radj) < config['entry_min_risk_adj_roll']:
                        continue

            trade = FlyTrade(
                fly_id=col,
                entry_date=dt,
                entry_level=level,
                entry_zscore=z,
                entry_vol=vol if not np.isnan(vol) else 0.0,
                entry_roll=roll if not np.isnan(roll) else 0.0,
                direction=direction,
            )
            open_trades.append(trade)
            open_fly_ids.add(col)
            n_open += 1

    # Close remaining at end
    last_date = bt_dates[-1]
    for trade in open_trades:
        trade.exit_date = last_date
        trade.exit_reason = 'end_of_backtest'
        col = trade.fly_id
        if col in fly_ts.columns and last_date in fly_ts.index:
            trade.exit_level = fly_ts.loc[last_date, col]
            trade.exit_zscore = zscore_ts.loc[last_date, col] if last_date in zscore_ts.index else np.nan
        trade.realized_pnl -= cost
        closed_trades.append(trade)

    return closed_trades, daily_pnl

print('Backtest engine defined.')

---
## 4. Run Backtest

In [ ]:
trades, daily_pnl = run_fly_backtest(
    fly_ts, zscore_ts, vol_ts, roll_ts, risk_adj_roll_ts, BACKTEST_CONFIG
)

cum_pnl = daily_pnl.cumsum()
drawdown = cum_pnl - cum_pnl.cummax()

# Metrics
clean = daily_pnl.dropna()
completed = [t for t in trades if t.exit_date is not None]
winning = [t for t in completed if t.realized_pnl > 0]
std_d = clean.std()

metrics = {
    'Total P&L (bp)': clean.sum(),
    'Trades': len(completed),
    'Win Rate': len(winning) / len(completed) if completed else 0,
    'Avg P&L (bp)': np.mean([t.realized_pnl for t in completed]) if completed else 0,
    'Avg Winner (bp)': np.mean([t.realized_pnl for t in winning]) if winning else 0,
    'Avg Loser (bp)': np.mean([t.realized_pnl for t in completed if t.realized_pnl <= 0]) if any(t.realized_pnl <= 0 for t in completed) else 0,
    'Sharpe': (clean.mean() / std_d * np.sqrt(252)) if std_d > 0 else 0,
    'Max DD (bp)': drawdown.min(),
    'Avg Holding (days)': np.mean([len(pd.bdate_range(t.entry_date, t.exit_date)) - 1 for t in completed]) if completed else 0,
}

print(f'=== {gap_label} Fly RV Backtest Results ===')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:25s} {v:>10.2f}')
    else:
        print(f'  {k:25s} {v:>10}')

---
## 5. P&L Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12), gridspec_kw={'height_ratios': [3, 1, 1]})

# Cumulative P&L
ax = axes[0]
ax.plot(cum_pnl.index, cum_pnl.values, color='tab:cyan', linewidth=1.5)
ax.fill_between(cum_pnl.index, 0, cum_pnl.values,
                where=cum_pnl.values >= 0, color='tab:green', alpha=0.15)
ax.fill_between(cum_pnl.index, 0, cum_pnl.values,
                where=cum_pnl.values < 0, color='tab:red', alpha=0.15)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title(f'{gap_label} Fly RV Backtest -- Cumulative P&L (bp) | Sharpe={metrics["Sharpe"]:.2f} | Win={metrics["Win Rate"]:.0%}',
             fontweight='bold')
ax.set_ylabel('Cumulative P&L (bp)')
ax.grid(True, alpha=0.3)

# Mark trade entries/exits
for t in completed:
    if t.entry_date in cum_pnl.index:
        color = 'green' if t.direction == 1 else 'red'
        ax.axvline(t.entry_date, color=color, alpha=0.1, linewidth=0.5)

# Daily P&L histogram
ax = axes[1]
colors = ['tab:green' if x >= 0 else 'tab:red' for x in daily_pnl.values]
ax.bar(daily_pnl.index, daily_pnl.values, color=colors, alpha=0.6, width=1)
ax.set_title('Daily P&L (bp)', fontweight='bold')
ax.set_ylabel('bp')
ax.grid(True, alpha=0.3)

# Drawdown
ax = axes[2]
ax.fill_between(drawdown.index, 0, drawdown.values, color='tab:red', alpha=0.4)
ax.plot(drawdown.index, drawdown.values, color='tab:red', linewidth=0.8)
ax.set_title(f'Drawdown (bp) | Max DD = {metrics["Max DD (bp)"]:.1f} bp', fontweight='bold')
ax.set_ylabel('Drawdown (bp)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Trade Log & Exit Reason Analysis

In [ ]:
trade_rows = []
for t in completed:
    holding = len(pd.bdate_range(t.entry_date, t.exit_date)) - 1 if t.exit_date else 0
    trade_rows.append({
        'Fly': t.fly_id,
        'Dir': 'BUY' if t.direction == 1 else 'SELL',
        'Entry': t.entry_date.strftime('%Y-%m-%d'),
        'Exit': t.exit_date.strftime('%Y-%m-%d') if t.exit_date else '',
        'Entry Z': round(t.entry_zscore, 2),
        'Exit Z': round(t.exit_zscore, 2) if t.exit_zscore and not np.isnan(t.exit_zscore) else None,
        'Entry Lvl': round(t.entry_level, 2),
        'Exit Lvl': round(t.exit_level, 2) if t.exit_level else None,
        'P&L (bp)': round(t.realized_pnl, 2),
        'Peak (bp)': round(t.peak_pnl, 2),
        'Trough (bp)': round(t.trough_pnl, 2),
        'Days': holding,
        'Exit Reason': t.exit_reason,
    })

trades_df = pd.DataFrame(trade_rows)
print(f'=== Trade Log ({len(trades_df)} trades) ===')
display(trades_df)

In [ ]:
if not trades_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Exit reason breakdown
    ax = axes[0]
    reason_counts = trades_df['Exit Reason'].value_counts()
    reason_counts.plot(kind='bar', ax=ax, color='tab:cyan', alpha=0.7)
    ax.set_title('Exit Reason Distribution', fontweight='bold')
    ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

    # P&L by exit reason
    ax = axes[1]
    reason_pnl = trades_df.groupby('Exit Reason')['P&L (bp)'].mean()
    colors = ['tab:green' if v >= 0 else 'tab:red' for v in reason_pnl.values]
    reason_pnl.plot(kind='bar', ax=ax, color=colors, alpha=0.7)
    ax.set_title('Avg P&L by Exit Reason (bp)', fontweight='bold')
    ax.set_ylabel('Avg P&L (bp)')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.grid(True, alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

    # P&L distribution
    ax = axes[2]
    trades_df['P&L (bp)'].hist(bins=20, ax=ax, color='tab:cyan', alpha=0.7, edgecolor='white')
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.axvline(trades_df['P&L (bp)'].mean(), color='tab:orange', linestyle='--',
              label=f'Mean={trades_df["P&L (bp)"].mean():.1f} bp')
    ax.set_title('P&L Distribution', fontweight='bold')
    ax.set_xlabel('P&L (bp)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary by exit reason
    print('\n=== P&L by Exit Reason ===')
    summary = trades_df.groupby('Exit Reason').agg(
        Count=('P&L (bp)', 'count'),
        Avg_PnL=('P&L (bp)', 'mean'),
        Total_PnL=('P&L (bp)', 'sum'),
        Win_Rate=('P&L (bp)', lambda x: (x > 0).mean()),
        Avg_Days=('Days', 'mean'),
    ).round(2)
    display(summary)

---
## 7. P&L by Fly Position

In [ ]:
if not trades_df.empty:
    fly_summary = trades_df.groupby('Fly').agg(
        Count=('P&L (bp)', 'count'),
        Total_PnL=('P&L (bp)', 'sum'),
        Avg_PnL=('P&L (bp)', 'mean'),
        Win_Rate=('P&L (bp)', lambda x: (x > 0).mean()),
    ).round(2).sort_values('Total_PnL', ascending=False)

    print('=== P&L by Fly ===')
    display(fly_summary)

    fig, ax = plt.subplots(figsize=(14, 5))
    colors = ['tab:green' if v >= 0 else 'tab:red' for v in fly_summary['Total_PnL'].values]
    fly_summary['Total_PnL'].plot(kind='bar', ax=ax, color=colors, alpha=0.7)
    ax.set_title('Total P&L by Fly (bp)', fontweight='bold')
    ax.set_ylabel('Total P&L (bp)')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.grid(True, alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

---
## 8. Sensitivity Analysis

Sweep key parameters to find robust settings.

In [ ]:
# Parameter sweep: entry z-score threshold
sweep_results = []

for z_thresh in [1.0, 1.25, 1.5, 1.75, 2.0, 2.5]:
    for max_hold in [10, 15, 22, 44]:
        for stop_sd in [1.5, 2.0, 3.0, None]:
            test_config = dict(BACKTEST_CONFIG)
            test_config['entry_min_zscore'] = z_thresh
            test_config['exit_max_holding_days'] = max_hold
            test_config['exit_stop_loss_sd'] = stop_sd

            try:
                t, d = run_fly_backtest(
                    fly_ts, zscore_ts, vol_ts, roll_ts, risk_adj_roll_ts, test_config
                )
                c = d.dropna()
                s = c.std()
                comp = [x for x in t if x.exit_date]
                win = [x for x in comp if x.realized_pnl > 0]
                sweep_results.append({
                    'Z Thresh': z_thresh,
                    'Max Hold': max_hold,
                    'Stop SD': stop_sd if stop_sd else 'None',
                    'Total PnL': round(c.sum(), 1),
                    'Trades': len(comp),
                    'Win Rate': round(len(win) / len(comp), 2) if comp else 0,
                    'Sharpe': round(c.mean() / s * np.sqrt(252), 2) if s > 0 else 0,
                    'Max DD': round((c.cumsum() - c.cumsum().cummax()).min(), 1),
                })
            except Exception:
                pass

sweep_df = pd.DataFrame(sweep_results)
if not sweep_df.empty:
    print(f'=== Sensitivity Sweep ({len(sweep_df)} combinations) ===')
    print('\nTop 10 by Sharpe:')
    display(sweep_df.nlargest(10, 'Sharpe'))

    print('\nTop 10 by Total P&L:')
    display(sweep_df.nlargest(10, 'Total PnL'))

    print('\nTop 10 by Win Rate (min 5 trades):')
    display(sweep_df[sweep_df['Trades'] >= 5].nlargest(10, 'Win Rate'))
else:
    print('No sweep results (check data availability)')

---
## 9. Carry Filter Impact

Compare with and without requiring carry alignment.

In [ ]:
# Without carry filter
cfg_no_carry = dict(BACKTEST_CONFIG)
cfg_no_carry['entry_require_carry'] = False
t1, d1 = run_fly_backtest(fly_ts, zscore_ts, vol_ts, roll_ts, risk_adj_roll_ts, cfg_no_carry)

# With carry filter
cfg_carry = dict(BACKTEST_CONFIG)
cfg_carry['entry_require_carry'] = True
t2, d2 = run_fly_backtest(fly_ts, zscore_ts, vol_ts, roll_ts, risk_adj_roll_ts, cfg_carry)

fig, ax = plt.subplots(figsize=(16, 6))
d1.cumsum().plot(ax=ax, label=f'No carry filter ({len(t1)} trades)', linewidth=1.5)
d2.cumsum().plot(ax=ax, label=f'With carry filter ({len(t2)} trades)', linewidth=1.5)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Carry Filter Impact on Cumulative P&L', fontweight='bold')
ax.set_ylabel('Cumulative P&L (bp)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for label, trades_list, daily in [('No carry', t1, d1), ('With carry', t2, d2)]:
    c = daily.dropna()
    s = c.std()
    comp = [t for t in trades_list if t.exit_date]
    win = [t for t in comp if t.realized_pnl > 0]
    sharpe = c.mean() / s * np.sqrt(252) if s > 0 else 0
    print(f'{label:12s}: PnL={c.sum():+.1f} bp | Trades={len(comp)} | WR={len(win)/len(comp):.0%} | Sharpe={sharpe:.2f}' if comp else f'{label}: no trades')

---
## 10. Exit Strategy Comparison

Compare different take-profit / stop-loss approaches.

In [ ]:
exit_configs = {
    'Mean Rev Only': {'exit_mean_reversion': True, 'exit_stop_loss_sd': None,
                      'exit_take_profit_zscore': None, 'exit_take_profit_bp': None,
                      'exit_stop_loss_bp': None, 'exit_max_holding_days': 44},
    'Mean Rev + Stop 2sd': {'exit_mean_reversion': True, 'exit_stop_loss_sd': 2.0,
                            'exit_take_profit_zscore': None, 'exit_take_profit_bp': None,
                            'exit_stop_loss_bp': None, 'exit_max_holding_days': 22},
    'TP z<0.5 + Stop 2sd': {'exit_mean_reversion': False, 'exit_stop_loss_sd': 2.0,
                             'exit_take_profit_zscore': 0.5, 'exit_take_profit_bp': None,
                             'exit_stop_loss_bp': None, 'exit_max_holding_days': 22},
    'TP +3bp + SL -5bp': {'exit_mean_reversion': False, 'exit_stop_loss_sd': None,
                           'exit_take_profit_zscore': None, 'exit_take_profit_bp': 3.0,
                           'exit_stop_loss_bp': -5.0, 'exit_max_holding_days': 22},
    'TP +5bp + SL -3bp': {'exit_mean_reversion': False, 'exit_stop_loss_sd': None,
                           'exit_take_profit_zscore': None, 'exit_take_profit_bp': 5.0,
                           'exit_stop_loss_bp': -3.0, 'exit_max_holding_days': 22},
    'Tight: TP z<0.3 + Stop 1.5sd': {'exit_mean_reversion': False, 'exit_stop_loss_sd': 1.5,
                                      'exit_take_profit_zscore': 0.3, 'exit_take_profit_bp': None,
                                      'exit_stop_loss_bp': None, 'exit_max_holding_days': 15},
}

fig, ax = plt.subplots(figsize=(18, 7))
exit_summary = []

for name, overrides in exit_configs.items():
    cfg = dict(BACKTEST_CONFIG)
    cfg.update(overrides)
    try:
        t, d = run_fly_backtest(fly_ts, zscore_ts, vol_ts, roll_ts, risk_adj_roll_ts, cfg)
        c = d.dropna()
        cum = c.cumsum()
        cum.plot(ax=ax, label=name, linewidth=1.5)
        s = c.std()
        comp = [x for x in t if x.exit_date]
        win = [x for x in comp if x.realized_pnl > 0]
        exit_summary.append({
            'Strategy': name,
            'Total PnL': round(c.sum(), 1),
            'Trades': len(comp),
            'Win Rate': f'{len(win)/len(comp):.0%}' if comp else 'N/A',
            'Sharpe': round(c.mean() / s * np.sqrt(252), 2) if s > 0 else 0,
            'Max DD': round((c.cumsum() - c.cumsum().cummax()).min(), 1),
            'Avg Hold': round(np.mean([len(pd.bdate_range(x.entry_date, x.exit_date))-1 for x in comp]), 1) if comp else 0,
        })
    except Exception as e:
        print(f'{name}: ERROR - {e}')

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Exit Strategy Comparison -- Cumulative P&L (bp)', fontweight='bold')
ax.set_ylabel('Cumulative P&L (bp)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n=== Exit Strategy Summary ===')
display(pd.DataFrame(exit_summary).set_index('Strategy'))